# Financial Risk Analysis with Python  
## Task 3: Customer Profile Building    

**Objective:**  
To segment customer accounts based on transaction activity, balance behavior, and transaction volume in order to build meaningful customer profiles and identify financially significant and at-risk account groups.


In [13]:
# Load cleaned transaction data for Task 3
import pandas as pd

df = pd.read_csv("cleaned_transactions.csv")

# Ensure transaction date is in datetime format
df['TransactionDate'] = pd.to_datetime(df['TransactionDate'])

df.head()


,TransactionID,CustomerID,AccountID,AccountType,TransactionType,Product,Firm,Region,Manager,TransactionDate,TransactionAmount,AccountBalance,RiskScore,CreditRating,TenureMonths,year,month
0,78,CUST1223,ACC33287,credit,withdrawal,Savings Account,Firm E,South,Manager 4,2023-06-01,81300.425190,40843.56193,0.330474,484,13,2023.0,2023-06
1,21,CUST8266,ACC58667,savings,withdrawal,Credit Card,Firm A,South,Manager 2,NaT,9269.640373,61183.03953,0.089688,836,18,NaN,NaN
2,176,CUST9420,ACC99117,credit,transfer,Home Loan,Firm A,East,Manager 1,NaT,28138.552650,85460.13405,0.340010,451,25,NaN,NaN
3,167,CUST5253,ACC10117,loan,payment,Mutual Fund,Firm D,Central,Manager 1,NaT,83943.556980,100525.35900,0.605383,487,13,NaN,NaN
4,46,CUST1223,ACC74631,savings,deposit,Credit Card,Firm A,East,Manager 4,2023-12-05,77104.456470,57425.69930,1.042441,393,10,2023.0,2023-12


### Activity Level Classification Rubric

Accounts are classified based on **average monthly transaction frequency** as follows:

- **High Activity:** 20 or more transactions per month  
- **Medium Activity:** 5 to 19 transactions per month  
- **Low Activity:** Fewer than 5 transactions per month

This rubric is defined to distinguish highly engaged customers from moderately active and inactive accounts.


In [14]:
# Calculate monthly transaction count per account
monthly_txn_count = (
    df.groupby(['AccountID', df['TransactionDate'].dt.to_period('M')])
      .size()
      .reset_index(name='txn_count')
)

# Calculate average monthly transactions per account
avg_monthly_txn = (
    monthly_txn_count.groupby('AccountID')['txn_count']
    .mean()
    .reset_index(name='avg_txn_per_month')
)

# Assign activity level based on rubric
def activity_level(x):
    if x >= 20:
        return 'High'
    elif x >= 5:
        return 'Medium'
    else:
        return 'Low'

avg_monthly_txn['activity_level'] = avg_monthly_txn['avg_txn_per_month'].apply(activity_level)

avg_monthly_txn.head()


,AccountID,avg_txn_per_month,activity_level
0,ACC10117,1.0,Low
1,ACC10996,1.0,Low
2,ACC11062,1.0,Low
3,ACC11188,1.0,Low
4,ACC11285,1.0,Low


In [16]:
# Segment customers by average account balance and transaction volume

# Calculate average balance per account
avg_balance = (
    df.groupby('AccountID')['AccountBalance']
      .mean()
      .reset_index(name='avg_balance')
)

# Calculate total transaction volume per account
txn_volume = (
    df.groupby('AccountID')['TransactionAmount']
      .sum()
      .reset_index(name='total_transaction_volume')
)

# Merge balance and transaction volume
customer_segments = avg_balance.merge(
    txn_volume, on='AccountID', how='inner'
)

customer_segments.head()


,AccountID,avg_balance,total_transaction_volume
0,ACC10117,94082.590727,599628.73770
1,ACC10996,72517.761136,506746.15806
2,ACC11062,73541.328302,247384.20924
3,ACC11188,51359.039950,127634.92588
4,ACC11285,79923.619870,169318.96953


In [19]:
# Recreate credit/debit classification (required for Task 3)
df['TransactionType'] = df['TransactionType'].str.lower().str.strip()

df['credit_debit'] = df['TransactionType'].map({
    'deposit': 'credit',
    'transfer': 'credit',
    'payment': 'debit',
    'withdrawal': 'debit'
})


In [20]:
# Create customer profiles as per Task 3

# --- 1. High-net inflow accounts ---
account_net_inflow = (
    df.groupby(['AccountID', 'credit_debit'])['TransactionAmount']
      .sum()
      .unstack(fill_value=0)
)

account_net_inflow['net_inflow'] = (
    account_net_inflow['credit'] - account_net_inflow['debit']
)

high_net_inflow_accounts = (
    account_net_inflow.sort_values('net_inflow', ascending=False)
    .reset_index()
)

# --- 2. High-frequency, low-balance accounts ---
# Transaction frequency per account
txn_frequency = (
    df.groupby('AccountID')
      .size()
      .reset_index(name='transaction_count')
)

# Average balance per account
avg_balance = (
    df.groupby('AccountID')['AccountBalance']
      .mean()
      .reset_index(name='avg_balance')
)

# Merge frequency and balance
freq_balance = txn_frequency.merge(avg_balance, on='AccountID')

high_freq_low_balance_accounts = freq_balance[
    (freq_balance['transaction_count'] >= 20) &
    (freq_balance['avg_balance'] < freq_balance['avg_balance'].median())
]

# --- 3. Negative or near-zero balance accounts ---
negative_near_zero_accounts = avg_balance[
    avg_balance['avg_balance'] <= 0
]

# View samples
high_net_inflow_accounts.head(), high_freq_low_balance_accounts.head(), negative_near_zero_accounts.head()


(credit_debit AccountID        credit         debit    net_inflow
 0             ACC99549  575752.64702   53636.57769  522116.06933
 1             ACC57700  328840.57400       0.00000  328840.57400
 2             ACC95774  313828.49425       0.00000  313828.49425
 3             ACC48501  462332.99591  151655.14972  310677.84619
 4             ACC74631  278462.66736  -12893.46263  291356.12999,
 Empty DataFrame
 Columns: [AccountID, transaction_count, avg_balance]
 Index: [],
 Empty DataFrame
 Columns: [AccountID, avg_balance]
 Index: [])